# CCIS 727 Assignment 7: CLIP prompt comparison

Zero-shot classification of 100 official CIFAR-10 **test** images (10 per class, seed 42) with frozen pretrained OpenAI CLIP ViT-B/32. A uses the class name; B uses `a photo of a {class}.`. Both conditions use the same images, image embeddings, preprocessing, tokenizer, and candidate class order. No training or prompt tuning is performed.

Run all cells from the repository root. Downloads are checksum-verified by torchvision and OpenAI CLIP. Dataset and checkpoint caches are excluded from Git. Cosine scores are unscaled similarities, not probabilities.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import clip
from torchvision.datasets import CIFAR10
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

SEED = 42
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = Path.cwd()
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
packages = ["torch", "torchvision", "clip", "numpy", "pandas", "matplotlib",
            "scikit-learn", "nbformat", "nbclient", "ipykernel", "reportlab", "Pillow"]
versions = {name: metadata.version(name) for name in packages}
hardware = {"device": DEVICE, "platform": platform.platform(),
            "processor": platform.processor(), "torch_threads": torch.get_num_threads(),
            "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
print("Python:", platform.python_version())
print("Hardware:", hardware)
display(pd.Series(versions, name="version").to_frame())

Python: 3.12.14
Hardware: {'device': 'cpu', 'platform': 'Linux-6.18.57-x86_64-with-glibc2.41', 'processor': '', 'torch_threads': 4, 'gpu': None}


,version
torch,2.5.1+cpu
torchvision,0.20.1+cpu
clip,1.0
numpy,2.2.6
pandas,2.2.3
matplotlib,3.10.3
scikit-learn,1.6.1
nbformat,5.10.4
nbclient,0.10.2
ipykernel,6.29.5


## 1. Fixed stratified test sample

Use NumPy `default_rng(42)`. In CIFAR-10 class order, choose ten distinct test indices per class without replacement, sort the ten indices within each class, and concatenate. This defines the sampling algorithm completely; the seed alone is insufficient. The training split is never loaded.

In [2]:
dataset = CIFAR10(root=ROOT / "data", train=False, download=True)
classes = dataset.classes
targets = np.asarray(dataset.targets)
rng = np.random.default_rng(SEED)
indices = np.concatenate([
    np.sort(rng.choice(np.flatnonzero(targets == label), size=10, replace=False))
    for label in range(10)
])
y_true = targets[indices]
assert len(dataset) == 10000
assert len(indices) == len(np.unique(indices)) == 100
assert np.array_equal(np.bincount(y_true, minlength=10), np.full(10, 10))
selection = pd.DataFrame({"sample_id": np.arange(100), "dataset_index": indices,
                          "true_label_id": y_true,
                          "true_label": [classes[y] for y in y_true]})
selection.to_csv(OUT / "selected_indices.csv", index=False)
display(selection)
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for label, ax in enumerate(axes.flat):
    row = np.flatnonzero(y_true == label)[0]
    ax.imshow(dataset[int(indices[row])][0], interpolation="nearest")
    ax.set_title(f"{classes[label]} (index {indices[row]})")
    ax.axis("off")
fig.suptitle("One selected test image per CIFAR-10 class")
fig.tight_layout()
fig.savefig(OUT / "class_examples.png", dpi=180, bbox_inches="tight")
plt.show()

URLError: <urlopen error Tunnel connection failed: 403 Forbidden>

## 2. Frozen CLIP encoders and cosine scores

Use official `clip.load("ViT-B/32")` preprocessing and `clip.tokenize`. Encode the 100 images **once** in batches of 16. Each condition encodes ten texts. L2-normalize each image and text vector; their matrix product is the 100×10 cosine matrix. The shared 512-dimensional space makes image–text dot products possible; contrastive pretraining aligns the modalities semantically (course reading, “CLIP,” pp. 1–2). Prediction is the largest cosine in each row. We do not multiply by CLIP's learned logit scale or apply softmax.

In [ ]:
model, preprocess = clip.load("ViT-B/32", device=DEVICE, jit=False,
                              download_root=str(ROOT / ".cache" / "clip"))
model.eval()
model.requires_grad_(False)
print("Official preprocessing:", preprocess)
prompts = {"A": classes, "B": [f"a photo of a {name}." for name in classes]}
display(pd.DataFrame(prompts))
started = time.perf_counter()
with torch.inference_mode():
    image_parts = []
    for start in range(0, len(indices), BATCH_SIZE):
        batch = torch.stack([preprocess(dataset[int(i)][0])
                             for i in indices[start:start + BATCH_SIZE]]).to(DEVICE)
        image_parts.append(model.encode_image(batch).float().cpu())
    image_embeddings = torch.cat(image_parts)
    text_embeddings = {condition: model.encode_text(clip.tokenize(text).to(DEVICE)).float().cpu()
                       for condition, text in prompts.items()}
image_unit = F.normalize(image_embeddings, dim=1)
text_unit = {condition: F.normalize(embeddings, dim=1)
             for condition, embeddings in text_embeddings.items()}
scores = {condition: (image_unit @ embeddings.T).numpy()
          for condition, embeddings in text_unit.items()}
inference_seconds = time.perf_counter() - started
assert image_embeddings.shape == (100, 512)
assert all(t.shape == (10, 512) for t in text_embeddings.values())
assert torch.allclose(image_unit.norm(dim=1), torch.ones(100), atol=1e-5)
assert all(torch.allclose(t.norm(dim=1), torch.ones(10), atol=1e-5) for t in text_unit.values())
for condition, matrix in scores.items():
    assert matrix.shape == (100, 10) and np.isfinite(matrix).all()
    assert np.abs(matrix).max() <= 1 + 1e-5
    print(f"{condition}: image shape {tuple(image_embeddings.shape)}, text shape "
          f"{tuple(text_embeddings[condition].shape)}, scores {matrix.shape}")
    frame = pd.DataFrame(matrix, columns=classes)
    frame.insert(0, "dataset_index", indices)
    frame.insert(0, "sample_id", np.arange(100))
    frame.to_csv(OUT / f"scores_{condition}.csv", index=False, float_format="%.9g")
    with pd.option_context("display.max_rows", 100, "display.max_columns", 12):
        display(frame)
v = image_embeddings[0]
w = text_embeddings["A"][0]
dot = torch.dot(v, w).item()
v_norm, w_norm = v.norm().item(), w.norm().item()
manual_cosine = dot / (v_norm * w_norm)
assert np.isclose(manual_cosine, scores["A"][0, 0], atol=1e-6)
print(f"Numerical example: image index {indices[0]}, A text '{classes[0]}'")
print(f"dot={dot:.8f}; image norm={v_norm:.8f}; text norm={w_norm:.8f}")
print(f"cosine = {dot:.8f} / ({v_norm:.8f} * {w_norm:.8f}) = {manual_cosine:.8f}")
print(f"Matrix entry: {scores['A'][0, 0]:.8f}")

## 3. Accuracy, confusion matrices, and paired predictions

Per-class accuracy is the fraction correct among the ten images of the true class (recall). Confusion matrix rows are true labels and columns are predictions, in the displayed CIFAR-10 order. The difference is B minus A in percentage points. The four correctness categories partition the same 100 images.

In [ ]:
predictions = {c: matrix.argmax(axis=1) for c, matrix in scores.items()}
correct = {c: pred == y_true for c, pred in predictions.items()}
accuracy = {c: float(ok.mean()) for c, ok in correct.items()}
per_class = pd.DataFrame({"class": classes, "n": 10,
    **{f"accuracy_{c}": [float(ok[y_true == label].mean()) for label in range(10)]
       for c, ok in correct.items()}})
per_class.to_csv(OUT / "per_class_accuracy.csv", index=False)
display(per_class)
for c in prompts:
    print(f"Condition {c}: {correct[c].sum()}/100 = {100 * accuracy[c]:.1f}%")
    cm = confusion_matrix(y_true, predictions[c], labels=np.arange(10))
    assert cm.shape == (10, 10) and np.array_equal(cm.sum(axis=1), np.full(10, 10))
    fig, ax = plt.subplots(figsize=(8, 7))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45, values_format="d")
    ax.set_title(f"Condition {c}: confusion matrix (n=100)")
    fig.tight_layout()
    fig.savefig(OUT / f"confusion_{c}.png", dpi=180, bbox_inches="tight")
    plt.show()
paired = {"both_correct": int((correct["A"] & correct["B"]).sum()),
          "A_only_correct": int((correct["A"] & ~correct["B"]).sum()),
          "B_only_correct": int((~correct["A"] & correct["B"]).sum()),
          "both_wrong": int((~correct["A"] & ~correct["B"]).sum())}
difference_pp = 100 * (accuracy["B"] - accuracy["A"])
assert sum(paired.values()) == 100
assert np.isclose(difference_pp, paired["B_only_correct"] - paired["A_only_correct"])
comparison = pd.DataFrame({"condition": ["A", "B"],
                           "prompt": ["{class}", "a photo of a {class}."],
                           "correct": [int(correct[c].sum()) for c in prompts],
                           "total": [100, 100],
                           "accuracy_percent": [100 * accuracy[c] for c in prompts]})
display(comparison)
comparison.to_csv(OUT / "accuracy_comparison.csv", index=False)
print(f"Accuracy difference (B - A): {difference_pp:+.1f} percentage points")
print("Paired correctness:", paired)
results = selection.copy()
for c in prompts:
    results[f"prediction_{c}_id"] = predictions[c]
    results[f"prediction_{c}"] = [classes[p] for p in predictions[c]]
    results[f"correct_{c}"] = correct[c]
    results[f"top1_cosine_{c}"] = scores[c].max(axis=1)
results["prediction_changed"] = predictions["A"] != predictions["B"]
results.to_csv(OUT / "predictions.csv", index=False, float_format="%.9g")
display(results)

## 4. Five-image paired error analysis

Select deterministically: changed predictions with an error first, then other errors, then unchanged correct predictions only if needed; break ties by sample order. Each row shows the **same** original image for A and B, its true label, and both top-three class/cosine lists. A change in the text changes its embedding, so the same image may have different class rankings. These five examples are selected for inspection, not an independent performance sample.

In [ ]:
changed = predictions["A"] != predictions["B"]
error = ~correct["A"] | ~correct["B"]
priority = np.where(changed & error, 0, np.where(error, 1, 2))
chosen = np.lexsort((np.arange(100), priority))[:5]
print(f"Available images with an error in either condition: {int(error.sum())}; "
      f"selected errors: {int(error[chosen].sum())}; "
      f"correct fillers: {int((~error[chosen]).sum())}.")
error_rows = []
fig, axes = plt.subplots(5, 3, figsize=(13, 12), gridspec_kw={"width_ratios": [1, 2, 2]})
for row, sample in enumerate(chosen):
    idx = int(indices[sample])
    axes[row, 0].imshow(dataset[idx][0], interpolation="nearest")
    axes[row, 0].set_title(f"Index {idx}; true: {classes[y_true[sample]]}")
    axes[row, 0].axis("off")
    record = {"sample_id": int(sample), "dataset_index": idx,
              "true_label": classes[y_true[sample]]}
    for col, c in enumerate(prompts, start=1):
        top3 = np.argsort(-scores[c][sample], kind="stable")[:3]
        text = [f"{rank}. {classes[k]}: {scores[c][sample, k]:.6f}"
                for rank, k in enumerate(top3, start=1)]
        axes[row, col].text(0, .8, f"Condition {c} ({'correct' if correct[c][sample] else 'wrong'})\n"
                            + "\n".join(text), va="top", fontsize=12)
        axes[row, col].axis("off")
        for rank, k in enumerate(top3, start=1):
            record[f"{c}_top{rank}_class"] = classes[k]
            record[f"{c}_top{rank}_cosine"] = float(scores[c][sample, k])
    error_rows.append(record)
fig.suptitle("Five identical images compared across both prompt conditions")
fig.tight_layout()
fig.savefig(OUT / "five_image_analysis.png", dpi=180, bbox_inches="tight")
plt.show()
error_table = pd.DataFrame(error_rows)
error_table.to_csv(OUT / "five_image_analysis.csv", index=False, float_format="%.9g")
display(error_table)

## 5. Concise report and run record

The PDF below is generated from this run's measured results. The report addresses all five Section 5 questions and cites the supplied course reading by section and PDF page. Results are generated directly from this run.

In [ ]:
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image

checkpoint = ROOT / ".cache" / "clip" / "ViT-B-32.pt"
checkpoint_sha256 = hashlib.file_digest(checkpoint.open("rb"), "sha256").hexdigest()
summary = {"seed": SEED, "batch_size": BATCH_SIZE, "n": 100,
           "accuracy": accuracy, "difference_B_minus_A_pp": difference_pp,
           "paired": paired, "changed_predictions": int(changed.sum()),
           "checkpoint_sha256": checkpoint_sha256,
           "inference_seconds": inference_seconds,
           "python": platform.python_version(), "hardware": hardware, "packages": versions,
           "report_status": "analysis draft based on measured results"}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
styles = getSampleStyleSheet()
story = []
def paragraph(text, style="BodyText"):
    story.append(Paragraph(escape(text), styles[style]))
    story.append(Spacer(1, 6))
paragraph("CCIS 727 Assignment 7: CLIP prompt comparison", "Title")
paragraph("Analysis draft based on the fixed evaluation sample. References to the course reading use PDF page numbers.")
paragraph("Method", "Heading2")
paragraph("We sampled 10 images without replacement per class from the official CIFAR-10 test split "
          "using NumPy default_rng(42), in class order, sorting indices within each class. "
          "The same 100 images were evaluated under A (class name only) and B (a photo of a {class}.). "
          "Pretrained OpenAI CLIP ViT-B/32 used its official preprocessing and tokenizer. Both encoders "
          "were frozen in evaluation mode with inference_mode (no gradients). Image embeddings were computed "
          "once in batches of 16. Each 512-dimensional image/text vector was L2-normalized; "
          "dot products produced two 100 x 10 cosine matrices. The maximum score gave the prediction.")
paragraph("Measured results", "Heading2")
paragraph(f"A: {int(correct['A'].sum())}/100 ({100*accuracy['A']:.1f}%). "
          f"B: {int(correct['B'].sum())}/100 ({100*accuracy['B']:.1f}%). "
          f"B minus A: {difference_pp:+.1f} percentage points. "
          f"Both correct: {paired['both_correct']}; A only correct: {paired['A_only_correct']}; "
          f"B only correct: {paired['B_only_correct']}; both wrong: {paired['both_wrong']}. "
          f"Predicted classes changed for {int(changed.sum())} images.")
table = Table([["True class", "A correct / 10", "B correct / 10"]] +
              [[classes[k], str(int(correct['A'][y_true == k].sum())),
                str(int(correct['B'][y_true == k].sum()))] for k in range(10)])
table.setStyle(TableStyle([("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
                          ("GRID", (0,0), (-1,-1), .3, colors.grey),
                          ("FONTSIZE", (0,0), (-1,-1), 9)]))
story.append(table)
paragraph("Interpretation and limitations", "Heading2")
paragraph(f"The net change is exactly B-only correct minus A-only correct "
          f"({paired['B_only_correct']} - {paired['A_only_correct']}). "
          "Only the prompt changed, so changes in rankings arise from the text representations. "
          "CLIP aligns image and language representations through contrastive pretraining [1]; "
          "adding photographic context may alter that alignment, but this experiment does not identify "
          "the causal role of individual words. Cosines are not calibrated probabilities. "
          "This balanced 100-image sample is small (each image changes overall accuracy by one point; "
          "each class image changes class accuracy by ten points), so these descriptive results do not "
          "establish performance on the full test set or statistical significance. No prompts were tuned to the sample.")
paragraph("CLIP versus a generative VLM", "Heading2")
paragraph("CLIP independently encodes an image and candidate texts into a shared 512-dimensional space. "
          "The shared dimension makes their dot products defined; contrastive pretraining encourages "
          "matching image-caption pairs to align more strongly than mismatched pairs [1, 4: CLIP, pp. 1-2]. "
          "In this lab, the pretrained vision encoder, text encoder and their learned projection "
          "parameters are frozen; class descriptions substitute for trained classifier weights. "
          "LLaVA instead uses a vision encoder, a learned visual projector/adapter mapping visual "
          "features to the language model embedding space, and an autoregressive language model "
          "to generate an answer conditioned on visual tokens and a question [3]. The adapter bridges "
          "representations; the language model supports open-ended textual answers rather than "
          "ranking this fixed ten-class candidate list [4: VLM Architectures, pp. 4-5; MLP Adapter, p. 6]. No generative VLM was run here.")
paragraph("Resolution, compression, and scope", "Heading2")
paragraph("Higher image resolution can preserve small objects or text, often at the cost of more visual "
          "tokens and computation. Compressing visual tokens can reduce cost but may discard spatial "
          "or fine-detail evidence [4: Dynamic Resolution, p. 4; Cross Attention Adapter, p. 5]. These are plausible trade-offs, not measured effects here: all inputs "
          "are 32 x 32 CIFAR-10 images enlarged using the same official CLIP preprocessing, with "
          "the same ViT-B/32 encoder and no varied resolution or compression setting. Upsampling "
          "does not restore missing source detail. We have not established whether similar images "
          "occurred in CLIP pretraining; the results measure transfer without task-specific training "
          "and do not prove unseen pretraining content. One additional controlled experiment would "
          "freeze the same two prompts and pipeline and repeat on a prespecified larger stratified "
          "sample from the remaining test images, with no overlap, to assess sampling variability.")
paragraph("Five-image inspection", "Heading2")
for record in error_rows:
    sample = record['sample_id']
    details = []
    for c in prompts:
        details.append(c + ": " + ", ".join(
            f"{record[f'{c}_top{r}_class']} ({record[f'{c}_top{r}_cosine']:.4f})" for r in range(1,4)))
    paragraph(f"Index {record['dataset_index']}; true {record['true_label']}. " + "; ".join(details) +
              f". A {'correct' if correct['A'][sample] else 'wrong'}; "
              f"B {'correct' if correct['B'][sample] else 'wrong'}.")
paragraph("These cases prioritize changed predictions and mistakes; their paired rankings are shown "
          "with the identical input image in outputs/five_image_analysis.png. They illustrate individual "
          "confusions and cannot represent overall error prevalence.")
paragraph("Reproducibility and sources", "Heading2")
paragraph(f"Python {platform.python_version()}; device {DEVICE}; batch size {BATCH_SIZE}; seed {SEED}. "
          "Exact package versions, checkpoint digest, hardware, indices, predictions and score matrices "
          "are saved alongside the notebook. Official preprocessing resizes to 224 with bicubic interpolation, "
          "center-crops to 224, converts to RGB/tensor and uses CLIP channel normalization.")
paragraph("[1] Radford et al. (2021), Learning Transferable Visual Models From Natural Language Supervision, "
          "ICML / PMLR 139, https://proceedings.mlr.press/v139/radford21a.html. "
          "Official implementation and pretrained weights: https://github.com/openai/CLIP, "
          "revision d05afc436d78f1c48dc0dbf8e5980a9d471f35f6.")
paragraph("[2] Krizhevsky (2009), Learning Multiple Layers of Features from Tiny Images. "
          "CIFAR-10: https://www.cs.toronto.edu/~kriz/cifar.html; official test split loaded with torchvision.datasets.CIFAR10.")
paragraph("[3] Liu et al. (2023), Visual Instruction Tuning, NeurIPS, "
          "https://arxiv.org/abs/2304.08485.")
paragraph("[4] Course reading, Vision Language Models.pdf, dated August 14, 2025: CLIP (pp. 1-2), Dynamic Resolution (p. 4), VLM Architectures (pp. 4-5), Cross Attention Adapter (p. 5), MLP Adapter (p. 6).")
SimpleDocTemplate(str(ROOT / "report.pdf"), rightMargin=48, leftMargin=48,
                  topMargin=40, bottomMargin=40).build(story)
print(json.dumps(summary, indent=2))
print("Saved report.pdf and all required experimental artifacts in outputs/.")